# Hands-on RAG for Education
## Notebook 3: From notebook to chatbot

ACM SIGCITE 2026 tutorial · Yong Zheng, Illinois Institute of Technology

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/academictutorials/academictutorials.github.io/blob/main/EdRAG/resources/materials/practice_offline/03_notebook_to_chatbot_colab.ipynb)

**Self-paced, after the tutorial.** You need a Google account and your own free Gemini API key, the same as for Notebooks 1 and 2.

In Notebooks 1 and 2 every answer came out of a code cell. Students do not use notebooks: they use a chat window, and they ask follow-up questions. This notebook turns each of the four approaches into a chatbot:

| Approach | Chat window | What you do here |
| --- | --- | --- |
| 1. NotebookLM | Built in | Share your notebook as a chat (no code) |
| 2. Gemini File Search | None: it is an API | Put a Gradio chat window in front of it (a few lines) |
| 3. AnythingLLM | Built in | See how it reaches users; optionally call its API from your own window |
| 4. Built from scratch | None: it is your code | Wrap the Notebook 2 pipeline in one `chat()` function, handle follow-up questions, and show the sources beside each answer |

The chat windows are made with [Gradio](https://www.gradio.app), an open-source Python library for small web apps. `gr.ChatInterface` turns any function `chat(message, history)` that returns text into a chat window.

**How to run:** click a cell and press **Shift + Enter**. Run the cells from top to bottom. Each chat window opens below the cell that launches it; close it with the cell that follows before you open the next one.

## 0. Setup

This cell installs the packages Colab does not have (about one minute), loads the helper file from Notebooks 1 and 2, and downloads the documents. Run it every time you open the notebook, because Colab starts each session on a fresh machine.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "html-to-markdown==3.18.4", "pypdf==6.20.1",
                "beautifulsoup4==4.15.0", "lxml==6.1.3", "openai==3.29.0", "google-genai==2.30.1",
                "sentence-transformers", "gradio>=6,<7"], check=True)
# The same library versions as Notebook 2, so the chunks match the vectors computed on the tutorial server

import re, html, time, pathlib, urllib.request, importlib
import numpy as np
import httpx
import gradio as gr
from bs4 import BeautifulSoup
from html_to_markdown import convert
from openai import OpenAI
from google import genai
from IPython.display import Markdown, display

# Helper code shared with Notebooks 1 and 2: downloads, the embedding model, and answer display
try:
    urllib.request.urlretrieve("https://academictutorials.github.io/EdRAG/resources/materials/practice_offline/edrag_utils.py",
                               "edrag_utils.py")
except Exception as exc:
    print("Could not refresh edrag_utils.py, using the copy already here:", exc)
import edrag_utils
importlib.reload(edrag_utils)
from edrag_utils import *

DOCUMENTS = {"ITM_Student_Handbook.pdf": "", "Tuition_Fees.html": ""}    # the two lab documents
download({**DOCUMENTS, "ITM_Student_Handbook.md": "converted/"})         # plus the handbook converted with Docling
USER = "colab"

print("\nGradio", gr.__version__, "| Ready.")

### 0.1 Your Gemini API key

Use the free key from Notebook 1 (**https://aistudio.google.com/apikey**). If you saved it in Colab's Secrets panel (key icon on the left) as `GEMINI_API_KEY` with notebook access turned on, it is read from there; otherwise you are asked to paste it.

> The free tier allows about 15 requests per minute. A chat turn uses one or two requests (one to rewrite a follow-up question, one to answer), so ask at a normal pace. When you go over the limit, the client waits and retries.

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get("GEMINI_API_KEY")
    print("Using the GEMINI_API_KEY secret from Colab.")
except Exception:
    GEMINI_KEY = getpass("Paste your Gemini API key: ").strip()

CHAT_MODEL = FS_MODEL = "gemini-3.5-flash-lite"
llm = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=GEMINI_KEY, max_retries=8)
client = genai.Client(api_key=GEMINI_KEY)      # for Gemini File Search (section 2)
print("Model:", CHAT_MODEL)

### 0.2 Two pieces every chatbot here shares

**The chat history.** Gradio passes your function the conversation so far as a list of messages, each with a `role` (`user` or `assistant`) and a `content`. In Gradio 6 the content is a list of blocks such as `{"type": "text", "text": "..."}`; older tutorials on the web show plain strings or `(user, bot)` pairs, which Gradio 6 no longer accepts. `turns()` reads either form.

**Follow-up questions.** In a notebook every question stood alone. In a chat, students write “Can my instructor help me instead?” after a question about Canvas. Searching the documents with that sentence alone finds the wrong passages, because the word *Canvas* is not in it. So before retrieval, the model rewrites the last message into a question that stands on its own.

> **Core idea:** a chatbot adds one step to the RAG pipeline: `question = rewrite_followup(message, history)`. Retrieval then uses `question`, not `message`.

In [ ]:
def text_of(content):
    # The plain text of one message's content: a string, a {"type": "text", "text": ...} block, or a list of blocks
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text") or ""
    if isinstance(content, (list, tuple)):
        return " ".join(text_of(block) for block in content)
    return str(getattr(content, "text", "") or "")


def turns(history, last=6):
    # The last few messages of a Gradio chat history as (role, text) pairs
    pairs = []
    for m in (history or [])[-last:]:
        role = m.get("role") if isinstance(m, dict) else getattr(m, "role", "")
        content = m.get("content") if isinstance(m, dict) else getattr(m, "content", "")
        pairs.append((role, text_of(content)))
    return pairs


REWRITE_PROMPT = """Rewrite the student's last message as one question that can be understood without the conversation.
Keep every name, number, and detail about the student's situation. If the message already stands on its own,
return it unchanged. Return only the question.

Conversation so far:
{conversation}

Last message: {message}"""


def rewrite_followup(message, history):
    # Core idea: turn a follow-up into a stand-alone question before retrieval (no model call on the first turn)
    past = turns(history)
    if not past:
        return message
    conversation = "\n".join(f"{role}: {text[:500]}" for role, text in past)
    reply = llm.chat.completions.create(model=CHAT_MODEL, temperature=0, max_tokens=120, messages=[
        {"role": "user", "content": REWRITE_PROMPT.format(conversation=conversation, message=message)}])
    return (reply.choices[0].message.content or "").strip() or message


history = [{"role": "user", "content": QUESTIONS["Q3"]},
           {"role": "assistant", "content": [{"type": "text", "text": "Contact the OTS Support Desk at "
            "supportdesk@illinoistech.edu or 312.567.DESK (3375). ITM faculty, teaching assistants, "
            "and staff cannot help with Canvas issues."}]}]
print("Follow-up:", "Can my instructor help me instead?")
print("Rewritten:", rewrite_followup("Can my instructor help me instead?", history))

---
## 1. Approach 1: NotebookLM — share the notebook as a chat

NotebookLM already is a chatbot, so “deploying” it means sharing it. Use the notebook you built in Notebook 1 (section 2):

1. Open the notebook in **https://notebooklm.google.com** and click **Share** (top right).
2. Under **Notebook access**, choose **Anyone with a link**.
3. Click **Copy link**, or **Copy link to chat view** for a page that shows only the chat.
4. Open the link in a private browser window to see what a student sees, and ask Q1 there.

**Before you share it with students**

* **Public links work only on personal Google accounts.** Google has turned public sharing off for Workspace Education and Enterprise accounts, so a notebook made with a school account may not offer **Anyone with a link**.
* **Viewers need a Google account**, and the documents stay with Google.
* **The chat view hides the sources but does not lock them.** According to Google's help page, viewers may still reach the sources; only paid accounts can share a notebook as *chat only*. Share only documents you would publish anyway.
* **You cannot change the prompt or the retrieval.** What you scored in Notebook 1 is what students get.

*Help page:* https://support.google.com/notebooklm/answer/16322204

---
## 2. Approach 2: Gemini File Search — your own chat window in front of an API

File Search has no window of its own, so we write one. The chat function has three lines that matter: rewrite the follow-up, ask File Search, and add the pages it cites.

### 2.1 Your document store

This reuses the store `edrag-colab` you built in Notebook 1. If it is gone, the cell builds it again (about 20 seconds).

In [ ]:
store_name = f"edrag-{USER}"
store = next((s for s in client.file_search_stores.list() if s.display_name == store_name), None)
if store is None:
    store = client.file_search_stores.create(config={"display_name": store_name})
    for file_name in DOCUMENTS:
        op = client.file_search_stores.upload_to_file_search_store(
            file=str(DATA / file_name), file_search_store_name=store.name, config={"display_name": file_name})
        while not op.done:
            time.sleep(3)
            op = client.operations.get(op)
        print("Indexed", file_name)
else:
    print("Reusing your store from Notebook 1")

### 2.2 The chat function and the window

`file_search_answer()` is `ask_file_search()` from Notebook 1, changed to return the answer instead of printing it. `chat_file_search()` is what Gradio calls for every message.

In [ ]:
def file_search_answer(question):
    # Ask File Search one question; return the answer and the (file, page) pairs it cites
    result = client.interactions.create(
        model=FS_MODEL, input=question,
        tools=[{"type": "file_search", "file_search_store_names": [store.name]}])
    answer, cites = "", set()
    for step in result.steps:
        if step.type == "model_output":
            for block in step.content:
                if block.type == "text":
                    answer += block.text
                    for a in block.annotations or []:
                        if a.type == "file_citation":
                            cites.add((a.file_name, getattr(a, "page_number", None)))
    return answer, sorted(cites, key=str)


def chat_file_search(message, history):
    question = rewrite_followup(message, history)          # 1. a stand-alone question
    answer, cites = file_search_answer(question)           # 2. File Search retrieves and answers
    cited = ", ".join(f"{f} p.{p}" if p else f for f, p in cites) or "nothing"
    searched = f"\n\n*Searched for:* {question}" if question != message else ""
    return f"{answer}\n\n**Cited:** {cited}{searched}"    # 3. the answer with its sources


fs_demo = gr.ChatInterface(fn=chat_file_search, title="ITM Advising Assistant (Gemini File Search)",
                           examples=[QUESTIONS["Q1"], QUESTIONS["Q3"]])
fs_demo.launch()

**Try it:** ask Q3, then “Can my instructor help me instead?”. The *Searched for* line shows the rewritten question. Then ask Q1 and “What about undergraduates?”.

> **Is this window private?** Not completely. On Colab, Gradio cannot serve the window only to you, so it also creates a temporary public address (`https://….gradio.live`, printed above the window). Anyone who has that address can use your chatbot, and your Gemini key pays for it. The address is long and random, but close the window when you are done:

In [ ]:
fs_demo.close()

---
## 3. Approach 3: AnythingLLM — already a chat app

AnythingLLM is a chatbot application, so there is nothing to wrap: you choose how it reaches users.

| How | Who can use it | What it takes |
| --- | --- | --- |
| **AnythingLLM Desktop** | Only you, on your computer | Install and add documents (Notebook 1, section 4.3) |
| **Docker on a server**, multi-user mode | Colleagues or students with accounts you create | A server with HTTPS, updates, and backups; this is how the tutorial ran it |
| **Embedded chat widget** (Docker version) | Visitors to a web page you control | Paste the script it generates into the page; the server must be reachable from that page |
| **Developer API** | Your own app or chat window | An API key from AnythingLLM's settings; see the optional cell below |

Colab cannot reach AnythingLLM on your own computer, so there is no demo to run here. The cell below shows the API route for readers who run AnythingLLM on a server they can reach. It follows the same pattern as section 2: rewrite the follow-up, ask the workspace in *query* mode, and list the sources. Leave the address empty and the cell skips itself.

> Keep the AnythingLLM API key out of shared notebooks: it gives full access to the workspace. Your server lists every endpoint at `/api/docs`.

In [ ]:
ANYTHINGLLM_URL = ""        # for example "https://anythingllm.your-school.edu"; leave empty to skip
ANYTHINGLLM_KEY = ""        # from AnythingLLM's settings (Developer API)
WORKSPACE = "edrag"         # the workspace's slug, shown in its address


def chat_anythingllm(message, history):
    question = rewrite_followup(message, history)
    r = httpx.post(f"{ANYTHINGLLM_URL}/api/v1/workspace/{WORKSPACE}/chat",
                   headers={"Authorization": f"Bearer {ANYTHINGLLM_KEY}"},
                   json={"message": question, "mode": "query"}, timeout=120)
    r.raise_for_status()
    data = r.json()
    titles = sorted({s.get("title", "?") for s in data.get("sources") or []})
    return f"{data.get('textResponse') or ''}\n\n**Sources:** {', '.join(titles) or 'none'}"


if ANYTHINGLLM_URL:
    allm_demo = gr.ChatInterface(fn=chat_anythingllm, title="ITM Advising Assistant (AnythingLLM)")
    allm_demo.launch()
else:
    print("Skipped: no AnythingLLM address set.")

---
## 4. Approach 4: Built from scratch — the full walk-through

This is the approach where you control every step, so it gets six small steps:

| Step | What you do |
| --- | --- |
| 4.1 | Load the chunks and vectors from Notebook 2 |
| 4.2 | Wrap retrieval and the prompt in one `chat()` function |
| 4.3 | Open a chat window with Gradio |
| 4.4 | Rewrite follow-up questions using the chat history |
| 4.5 | Show the retrieved chunks and sources beside each answer |
| 4.6 | Optional: share the window with a password, then close it |

### 4.1 Load the chunks and vectors from Notebook 2

This cell repeats the Notebook 2 code without the explanations: convert both documents to Markdown, fix the merged cells, clean the Markdown, cut it into heading-based chunks (preset C), and look up the chunk vectors computed on the tutorial server. If you changed a function in Notebook 2 and want your version here, paste it into this cell.

In [ ]:
def main_content(page_html):
    # Keep the page title and the <article> body; navigation, footer, and scripts are dropped
    soup = BeautifulSoup(page_html, "lxml")
    title = soup.select_one("h1").get_text(" ", strip=True)
    return BeautifulSoup(f"<h1>{title}</h1>{soup.select_one('article')}", "lxml")


def html_to_md(soup):
    # html-to-markdown 3.x returns an object; .content holds the Markdown text
    return convert(str(soup)).content


def annotate_merged_cells(soup):
    # Write out what each merged cell covers, and give each covered row a cell that points back
    for cell in soup.select("td[rowspan], th[rowspan]"):
        row = cell.find_parent("tr")
        covered = row.find_next_siblings("tr")[:int(cell["rowspan"]) - 1]
        label = lambda r: " ".join(r.find(["th", "td"]).get_text(" ").split())
        value = " ".join(cell.get_text(" ").split())
        first, others = label(row), ", ".join(label(r) for r in covered)
        col = row.find_all(["th", "td"], recursive=False).index(cell)
        cell.string = f"{value} (combined: {first} and {others})"
        del cell["rowspan"]
        for other in covered:
            filler = soup.new_tag("td")
            filler.string = f"(included in {first})"
            cells = other.find_all(["th", "td"], recursive=False)
            cells[col].insert_before(filler) if col < len(cells) else other.append(filler)
    return soup


def clean_markdown(text):
    # Drop the table of contents and image placeholders; fix entities, private-font bullets, and padded tables
    lines = []
    for line in text.splitlines():
        if re.search(r"\.{10,}", line):
            continue
        if line.strip() == "<!-- image -->":
            continue
        line = html.unescape(line)
        line = re.sub(r"[\ue000-\uf8ff]", "", line)
        if line.lstrip().startswith("|"):
            line = re.sub(r" {2,}", " ", line)
            if re.fullmatch(r"\|(\s*:?-+:?\s*\|)+", line.strip()):
                line = re.sub(r"-{3,}", "---", line)
        lines.append(line)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines))


def section_chunk(path, piece):
    # Every chunk starts with its section path
    return f"{path}\n\n{piece}"


HANDBOOK, TUITION = "ITM_Student_Handbook.pdf", "Tuition_Fees.html"
handbook_md = clean_markdown((DATA / "ITM_Student_Handbook.md").read_text(encoding="utf-8"))
page_html = (DATA / "Tuition_Fees.html").read_text(encoding="utf-8")
tuition_md = clean_markdown(html_to_md(annotate_merged_cells(main_content(page_html))))

CHUNKS = (chunk_by_heading(handbook_md, "ITM Graduate Student Handbook", HANDBOOK, section_chunk)
          + chunk_by_heading(tuition_md, "Tuition, Fees, and Costs", TUITION, section_chunk))
E, info = embed([c["text"] for c in CHUNKS])            # one row of 384 numbers per chunk
print(f"{len(CHUNKS)} chunks | matrix {E.shape[0]} x {E.shape[1]} | {info['cached']} vectors from the cache")

If the number of vectors from the cache is close to the number of chunks, the chunks match Notebook 2. If it is much lower, the embedding model computes the rest on the Colab machine, which takes a few minutes on the free CPU.

### 4.2 Wrap retrieval and the prompt in one `chat()` function

`search()` and the prompt are the ones from Notebook 2 (preset C, top 5, and the grounded prompt with the sentence you changed in step 4). `answer_with_sources()` is `answer()` from Notebook 2, changed to return the text instead of printing it.

This first `chat()` ignores the history: each message is treated as a new question.

In [ ]:
BEST_PROMPT = """You answer questions from university students using only the numbered passages below.

Rules:
1. Use only information in the passages. Do not add facts from memory.
2. After each sentence, cite the passages you used, for example [1] or [2][3].
3. If the passages do not contain the answer, say "I could not find this in the provided documents." Then give everything the passages do say that could help; if it differs for different groups of students, such as program level or visa status, give it for each group.
4. If the answer depends on the student's situation, such as program level or visa status, answer for each case.

{passages}

Question: {question}"""


def search(question, k=5):
    # Embed the question and return the k best (score, chunk) pairs: one matrix product, as in Notebook 2
    q, _ = embed([question], is_query=True)
    scores = E @ q[0]
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), CHUNKS[i]) for i in top]


def answer_with_sources(question, hits, prompt=BEST_PROMPT):
    # Number the retrieved chunks, put them into the prompt, and return the model's answer
    passages = "\n\n".join(f"[{i}] {c['text']}" for i, (_, c) in enumerate(hits, 1))
    reply = llm.chat.completions.create(model=CHAT_MODEL, temperature=0, max_tokens=700, messages=[
        {"role": "user", "content": prompt.format(passages=passages, question=question)}])
    return reply.choices[0].message.content


def chat(message, history):
    # Version 1: every message is a new question; the history is not used yet
    hits = search(message, k=5)
    return answer_with_sources(message, hits)


show_answer(chat(QUESTIONS["Q3"], []))

### 4.3 Open a chat window with Gradio

One line turns `chat()` into a chat window. Ask Q3 in it, then the follow-up “Can my instructor help me instead?”, and look at the second answer closely.

In [ ]:
demo = gr.ChatInterface(fn=chat, title="ITM Advising Assistant (built from scratch)",
                        examples=[QUESTIONS["Q3"], "Can my instructor help me instead?"])
demo.launch()

In [ ]:
demo.close()      # close the window before the next step (see the note in section 2 on the public address)

### 4.4 Rewrite follow-up questions using the chat history

Why did the follow-up go wrong? Look at what retrieval finds for the follow-up as written, and for the rewritten question.

In [ ]:
follow_up = "Can my instructor help me instead?"
print("Retrieved for the follow-up as written:")
show_hits(search(follow_up, k=3))

standalone = rewrite_followup(follow_up, history)       # `history` is the Q3 conversation from section 0.2
print("\nRewritten:", standalone)
show_hits(search(standalone, k=3))

The follow-up alone does not mention Canvas, so retrieval has to guess what it is about. Check whether the first list contains the *Canvas and Online Courses* chunk; the second list should have it near the top. The right answer depends on that chunk, because it is the one that says ITM faculty, teaching assistants, and staff cannot help with Canvas.

> **Core idea:** version 2 of `chat()` differs from version 1 in one line. This is the function on the tutorial slide.

In [ ]:
def chat(message, history):
    # Version 2: rewrite the follow-up, then retrieve and answer as before
    question = rewrite_followup(message, history)
    hits = search(question, k=5)
    return answer_with_sources(question, hits)


show_answer(chat(follow_up, history))

**Change one setting:** `turns()` passes the last 6 messages to the rewrite step. A long conversation that changes topic can mislead it. In section 0.2, try `last=2` and ask a follow-up to a follow-up.

### 4.5 Show the retrieved chunks and sources beside each answer

In Module 3 we argued that an answer should come with its evidence. A chat window can show the retrieved chunks next to the answer, so a student or an advisor can check it. `gr.ChatInterface` cannot add a side panel, so we lay out the window ourselves with `gr.Blocks`: the chat on the left, the chunks on the right. `respond()` does the same three steps as `chat()` and also returns the sources.

In [ ]:
NO_SOURCES = "*The chunks retrieved for each answer appear here.*"


def format_sources(question, hits):
    # One short entry per retrieved chunk: its number in the answer, section, score, and the start of its text
    lines = [f"**Searched for:** {question}"]
    for i, (score, c) in enumerate(hits, 1):
        body = c["text"].split("\n\n", 1)[-1]                   # the text after the section path
        excerpt = " ".join(body.split())[:220]
        lines.append(f"**[{i}]** {c['section'] or c['source']}  \n*score {score:.2f}* · {excerpt} …")
    return "\n\n".join(lines)


def respond(message, history):
    question = rewrite_followup(message, history)
    hits = search(question, k=5)
    reply = answer_with_sources(question, hits)
    history = list(history or []) + [{"role": "user", "content": message},
                                     {"role": "assistant", "content": reply}]
    return "", history, format_sources(question, hits)    # clear the box, update the chat, update the panel


with gr.Blocks(title="ITM Advising Assistant") as demo:
    gr.Markdown("### ITM Advising Assistant\nAnswers come only from the ITM Graduate Student Handbook and the "
                "Tuition, Fees, and Costs page. This is a teaching demo, not an official advisor: "
                "check every answer against the sources on the right.")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Chat", height=430)
            box = gr.Textbox(placeholder="Ask a question and press Enter", show_label=False)
            gr.Examples([QUESTIONS["Q1"], "What about undergraduates?", QUESTIONS["Q4"]], inputs=box)
            new_chat = gr.Button("New conversation")
        with gr.Column(scale=2):
            sources = gr.Markdown(NO_SOURCES)
    box.submit(respond, inputs=[box, chatbot], outputs=[box, chatbot, sources])
    new_chat.click(lambda: ([], NO_SOURCES), inputs=None, outputs=[chatbot, sources])

demo.launch()

**Try it:** ask Q1, then “What about undergraduates?”, then Q4 followed by “What if this is my first semester?”. For each answer, check that every `[n]` points to a chunk on the right that really says it.

In [ ]:
demo.close()

### 4.6 Optional: share the window with a password, then close it

The public address that Gradio prints can go to a colleague. Before you send it:

* **Use only public documents.** Everyone who opens the window can read whatever the chatbot retrieves.
* **Add a password** with `auth`, as below; Gradio then asks for it before showing the window.
* **Remember who pays.** Every question runs on your Gemini key and counts against your quota.
* **The address is temporary.** It stops working when you close the window, when the Colab session ends, or after Gradio's time limit (the message above the window says how long).

In [ ]:
SHARE_USER, SHARE_PASSWORD = "advisor", "change-me-please"     # pick your own before sharing
demo.launch(auth=(SHARE_USER, SHARE_PASSWORD))

In [ ]:
demo.close()      # run this when you are done

---
## 5. From a demo to a service

The windows in this notebook are demos: one person's key, a temporary address, no logins, and no records. Running a chatbot for real students needs more, and most of it is not code:

| A real service needs | Why | In this tutorial |
| --- | --- | --- |
| Keys kept on a server | A key in a browser or a shared notebook can be copied | The tutorial gateway held all keys; students never saw them |
| Sign-in through the institution | Know who asked, and keep out everyone else | JupyterHub with a shared password |
| Limits per user | One user or script should not use up the budget | 120 requests per minute and $2 per person |
| Logs that someone reviews | Find wrong answers and new questions | Gateway logs |
| A process to update the documents | Policies change every year; old chunks give old answers | Not covered |
| Care with student data | Records such as visa status or grades fall under FERPA | Public documents only |
| A way to reach a person | Uncertain or high-stakes questions belong to an advisor | Not covered |
| The test set, run again after every change | A change that fixes one question can break another | Notebook 2, step 4 |

Each approach covers a different part of this table for you: NotebookLM handles hosting and sign-in but nothing else; AnythingLLM gives accounts and logs on a server you run; a chatbot built from scratch gives you every choice and every task.

**Questions to think about**

* Which approach would you put in front of your own students, and which row of the table decides it?
* The rewrite step is a second model call that can also make mistakes. How would you test it?
* What should the window show when retrieval finds nothing with a high score?